# Randomized optimal stopping and reinforcement learning

This notebook is the entry point for the project. It compares the exploratory HJB finite-difference solver, Longstaff–Schwartz Monte Carlo, and neural fitted policy iteration. See `README.md` for the model, assumptions, and limitations.

Reference: Yuchao Dong (2023), [arXiv:2208.02409](https://arxiv.org/abs/2208.02409).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.finite_difference import solve_exploratory_put
from src.longstaff_schwartz import longstaff_schwartz_put
from src.simulation import simulate_gbm, put_payoff

## Model parameters

The baseline parameters below are illustrative. The exploratory value at positive temperature is not directly the same quantity as the classical American put price.

In [ ]:
S0 = 40.0
K = 40.0
r = 0.06
sigma = 0.40
T = 1.0
temperature = 0.1
seed = 1234

## 1. Exploratory HJB: implicit finite differences

The Hamiltonian maximizer is `pi* = exp((g - V) / temperature)`. The solver uses a truncated log-price domain, upwind drift and implicit time stepping.

In [ ]:
fd = solve_exploratory_put(
    s0=S0, strike=K, rate=r, volatility=sigma, maturity=T,
    temperature=temperature, space_steps=240, time_steps=240,
)
print(f"Exploratory value at S0={S0:g}: {fd['price']:.6f}")

plt.plot(fd['stock'], fd['values'][0])
plt.xlabel('Underlying price')
plt.ylabel('Exploratory value at t=0')
plt.title('Finite-difference HJB solution')
plt.grid(True)
plt.show()

## 2. Classical Bermudan put: Longstaff–Schwartz

The regression is fitted on in-the-money paths using least squares. Its reported standard error is conditional on the fitted exercise policy; use a separate sample for policy evaluation in a research-grade experiment.

In [ ]:
ls = longstaff_schwartz_put(
    s0=S0, strike=K, rate=r, volatility=sigma, maturity=T,
    steps=50, paths=50_000, seed=seed,
)
print(f"Bermudan put estimate: {ls['price']:.6f}")
print(f"Conditional MC standard error: {ls['standard_error']:.6f}")
print(f"95% CI: {ls['confidence_interval_95']}")

## 3. Neural fitted policy iteration (optional PyTorch)

This section is intentionally opt-in because neural training is computationally more expensive and experimental. Install with `pip install -e '.[rl]'`. The discrete survival factor is protected by an intensity cap; this is a numerical safeguard.

In [ ]:
# Uncomment to train and evaluate the neural policy.
# from src.reinforcement_learning import train_policy_iteration, evaluate_policy
# model = train_policy_iteration(
#     s0=S0, strike=K, rate=r, volatility=sigma, maturity=T,
#     steps=30, batch_size=512, iterations=300,
#     temperature=temperature, seed=seed,
# )
# evaluation = evaluate_policy(
#     model, s0=S0, strike=K, rate=r, volatility=sigma,
#     maturity=T, steps=30, paths=20_000, seed=seed + 1,
# )
# print(evaluation)

## Reproducibility and interpretation

- Refine the PDE space/time grids and spatial domain before interpreting convergence.
- Increase Monte Carlo paths and use independent policy-fitting/evaluation samples.
- Run multiple neural seeds and report uncertainty.
- Keep classical American/Bermudan prices separate from the entropy-regularized exploratory value at positive temperature.

Run the unit tests from the repository root with `pytest -q`.